<a href="https://colab.research.google.com/github/guilhermelaviola/SRTTranslator/blob/main/WordOriginDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Word Origin Detector**
This following script iterates through the subtitles, cleans out punctuation, and categorizes words based on the detected language.

In [6]:
pip install langid pysrt

In [7]:
import pysrt
import langid
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
# Stopwords setup:
es_stops = set(stopwords.words('spanish'))
pt_stops = set(stopwords.words('portuguese'))

# Isolating language-specific markers:
exclusive_es_stops = es_stops - pt_stops
exclusive_pt_stops = pt_stops - es_stops

# Character-level regex (Spanish unique chars vs Portuguese unique chars):
spanish_chars = re.compile(r'[ñÑ¿¡]')
portuguese_chars = re.compile(r'[çÇãõÃÕêÊ]')

# 3. Processing the SRT:
subs = pysrt.open('/content/File.srt', encoding='utf-8')

spanish_found = set()
portuguese_found = set()

for sub in subs:
    # Cleaning text and splitting into words:
    text = re.sub(r'<[^>]*>', '', sub.text)
    words = re.findall(r'\b\w+\b', text.lower())

    for word in words:
        # Checking by stopword list:
        if word in exclusive_es_stops:
            spanish_found.add(word)
        elif word in exclusive_pt_stops:
            portuguese_found.add(word)

        # Checking for unique characters (for example, words with 'ñ'):
        if spanish_chars.search(word):
            spanish_found.add(word)
        elif portuguese_chars.search(word):
            portuguese_found.add(word)

In [11]:
# Exporting to TXT files:
def save_to_txt(filename, word_set, label):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"--- {label} --- \n")
        # Sorting alphabetically makes the list much easier to scan
        for word in sorted(list(word_set)):
            f.write(f"{word}\n")
    print(f'File saved: {filename}')

save_to_txt('detected_spanish.txt', spanish_found, "Spanish Words/Markers")
save_to_txt('detected_portuguese.txt', portuguese_found, "Portuguese Words/Markers")

File saved: detected_spanish.txt
File saved: detected_portuguese.txt
